# Bootstrap ppxf Errors — z = 0.67564

Parallel run of the bootstrap error estimation with the deflector redshift
set to z = 0.67564 (from redshift verification / line fitting) instead of
the default z = 0.67511 (from ppxf). Tests whether the ~50 km/s velocity
offset in the input redshift affects the measured velocity dispersion.

Compare results to `03_bootstrap_ppxf_errors.ipynb` (z = 0.67511).

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import importlib
sys.path.insert(0, '..')
import scripts.bootstrap_ppxf as bootstrap_ppxf
importlib.reload(bootstrap_ppxf)
from scripts.bootstrap_ppxf import (
    setup_ppxf_inputs, run_bootstrap, compute_local_residual_scaling
)

plt.rcParams['figure.facecolor'] = 'white'
plt.rc('font', family='serif', size=14)
plt.rc('axes', linewidth=1.5, labelsize=16)
plt.rc('xtick', labelsize=14, direction='in')
plt.rc('ytick', labelsize=14, direction='in')

ifu_file = '../Nov17_2025_DESJ0206_RL_combined_icubes_wcs.fits'
results_dir = '../results'

Z_NEW = 0.67564
Z_OLD = 0.67511
print(f'Testing z = {Z_NEW} vs default z = {Z_OLD}')
print(f'Delta z = {Z_NEW - Z_OLD:.5f} ({(Z_NEW - Z_OLD) * 299792.458 / (1 + Z_OLD):.1f} km/s rest-frame)')

## 2. Quick test

In [ ]:
# Quick test with N=50 at selected degrees
test_new = run_bootstrap(
    ifu_file=ifu_file, sps_name='fsps', results_dir=results_dir,
    degrees=np.array([4, 10, 16, 20]),
    n_bootstrap=50, seed=42, save=False, z=Z_NEW
)
print('\nQuick test sigma (z=0.67564):')
for j, deg in enumerate(test_new['degrees']):
    print(f"  deg={deg}: sigma = {test_new['sigma_p50'][j]:.1f} "
          f"-{test_new['sigma_boot_err_lo'][j]:.1f} "
          f"+{test_new['sigma_boot_err_hi'][j]:.1f} km/s")

## 3. Full run (all templates, N=500)

In [ ]:
# Full bootstrap at z = 0.67564 for all three template libraries
results_new = {}
for sps in ['fsps', 'emiles', 'xsl']:
    print(f"\n{'=' * 60}")
    print(f"Bootstrap {sps} at z = {Z_NEW}")
    print(f"{'=' * 60}")
    results_new[sps] = run_bootstrap(
        ifu_file=ifu_file, sps_name=sps, results_dir=results_dir,
        n_bootstrap=500, seed=42, save=True,
        z=Z_NEW, save_suffix='z067564'
    )

## 4. Load z=0.67511 results for comparison

In [ ]:
# Load the default z=0.67511 bootstrap results
results_old = {}
for sps in ['fsps', 'emiles', 'xsl']:
    path = f'{results_dir}/ppxf_bootstrap_errors_{sps}.npz'
    try:
        results_old[sps] = dict(np.load(path, allow_pickle=True))
        print(f'Loaded {sps} z=0.67511 results: {path}')
    except FileNotFoundError:
        print(f'WARNING: {path} not found — run 03_bootstrap_ppxf_errors.ipynb first')

## 5. Comparison: sigma vs degree

In [ ]:
colors = {'fsps': 'C0', 'emiles': 'C1', 'xsl': 'C2'}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: sigma vs degree for both redshifts
ax = axes[0]
for sps in ['fsps', 'emiles', 'xsl']:
    rn = results_new[sps]
    ax.fill_between(rn['degrees'], rn['sigma_p16'], rn['sigma_p84'],
                    alpha=0.15, color=colors[sps])
    ax.plot(rn['degrees'], rn['sigma_original'], '-o', color=colors[sps],
            markersize=4, label=f'{sps} z={Z_NEW}')
    
    if sps in results_old:
        ro = results_old[sps]
        ax.plot(ro['degrees'], ro['sigma_original'], '--s', color=colors[sps],
                markersize=3, alpha=0.6, label=f'{sps} z={Z_OLD}')

ax.set_xlabel('Additive Polynomial Degree')
ax.set_ylabel(r'$\sigma$ (km/s)')
ax.set_title(r'$\sigma$ vs degree — solid: z=0.67564, dashed: z=0.67511')
ax.legend(fontsize=9, ncol=2)
ax.grid(alpha=0.3)

# Right: delta sigma (new - old)
ax = axes[1]
for sps in ['fsps', 'emiles', 'xsl']:
    if sps not in results_old:
        continue
    rn = results_new[sps]
    ro = results_old[sps]
    # Match degrees
    common_degs = np.intersect1d(rn['degrees'], ro['degrees'])
    idx_n = [np.where(rn['degrees'] == d)[0][0] for d in common_degs]
    idx_o = [np.where(ro['degrees'] == d)[0][0] for d in common_degs]
    
    delta = rn['sigma_original'][idx_n] - ro['sigma_original'][idx_o]
    ax.plot(common_degs, delta, '-o', color=colors[sps], markersize=4, label=sps)

ax.axhline(0, color='k', ls='--', lw=0.8)
ax.set_xlabel('Additive Polynomial Degree')
ax.set_ylabel(r'$\Delta\sigma$ (km/s)')
ax.set_title(r'$\sigma$(z=0.67564) $-$ $\sigma$(z=0.67511)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{results_dir}/figures/sigma_z_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Comparison: redshift distributions

In [ ]:
# Redshift vs degree for both input z values
fig, ax = plt.subplots(figsize=(12, 6))

for sps in ['fsps', 'emiles', 'xsl']:
    rn = results_new[sps]
    ax.fill_between(rn['degrees'], rn['z_p16'], rn['z_p84'],
                    alpha=0.15, color=colors[sps])
    ax.plot(rn['degrees'], rn['z_p50'], '-o', color=colors[sps],
            markersize=3, label=f'{sps} z_init={Z_NEW}')
    
    if sps in results_old and 'z_p50' in results_old[sps]:
        ro = results_old[sps]
        ax.plot(ro['degrees'], ro['z_p50'], '--s', color=colors[sps],
                markersize=3, alpha=0.6, label=f'{sps} z_init={Z_OLD}')

ax.axhline(Z_NEW, color='red', ls=':', lw=1, alpha=0.5, label=f'z={Z_NEW}')
ax.axhline(Z_OLD, color='blue', ls=':', lw=1, alpha=0.5, label=f'z={Z_OLD}')
ax.set_xlabel('Additive Polynomial Degree')
ax.set_ylabel('Fitted Redshift')
ax.set_title('Bootstrap redshift vs degree — two input z values')
ax.legend(fontsize=9, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Summary table

In [ ]:
# Summary at degree=16 for all templates, both redshifts
deg_report = 16
print(f"Comparison at degree={deg_report}")
print(f"{'Template':>8} {'z_init':>8} {'sigma':>7} {'-err':>6} {'+err':>6} {'V':>7} {'z_fit':>8}")
print('-' * 55)

for sps in ['fsps', 'emiles', 'xsl']:
    for label, res_dict, z_val in [('old', results_old, Z_OLD), ('new', results_new, Z_NEW)]:
        if sps not in res_dict:
            continue
        res = res_dict[sps]
        idx = np.where(res['degrees'] == deg_report)[0]
        if len(idx) == 0:
            continue
        idx = idx[0]
        print(f"{sps:>8} {z_val:8.5f} {res['sigma_original'][idx]:7.1f} "
              f"{res['sigma_boot_err_lo'][idx]:6.1f} {res['sigma_boot_err_hi'][idx]:6.1f} "
              f"{res['V_original'][idx]:7.1f} {res['z_p50'][idx]:8.5f}")
    print()